# Part 4 — Medallion persistence to HDFS

Extends the Part 3 streaming pipeline by replacing the **console** sinks with **HDFS Parquet** sinks in a 3-layer Medallion architecture:

| Layer  | Content                                          | Partitioning          |
|--------|--------------------------------------------------|-----------------------|
| Bronze | Raw parsed ticks (no cleaning)                   | `ingest_date`         |
| Silver | Cleaned ticks (after outlier filter)             | `ingest_date, symbol` |
| Gold   | Finalised 1-min moving averages (append mode)    | `window_date`         |

All three layers are written as **Parquet** (columnar, compressed, schema-embedded), each with its own `checkpointLocation` in HDFS so the queries can be killed and resumed independently.

**Prerequisite.** Run `02_producer_binance.ipynb` in another tab so ticks keep flowing into the `financial_data` topic while this consumer runs.

## 1. Spark session

In [ ]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("MedallionWriter")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0")
    .config("spark.sql.session.timeZone", "UTC")
    # Use the HDFS namenode from the compose network
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version, "ready")

## 2. Kafka → typed stream (same as Part 3)

We read from Kafka and apply the strict schema. `parsed` holds the raw typed stream; `clean` is `parsed` filtered through the outlier guard-rails.

In [ ]:
from pyspark.sql.functions import col, from_json, from_unixtime, lit, when, to_date
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

SCHEMA = StructType([
    StructField("symbol",    StringType(), False),
    StructField("price",     DoubleType(), False),
    StructField("volume",    DoubleType(), False),
    StructField("timestamp", LongType(),   False),
])

raw = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "financial_data")
    .option("startingOffsets", "latest")
    .load()
)

parsed = (
    raw.selectExpr("CAST(value AS STRING) AS json_str")
       .select(from_json(col("json_str"), SCHEMA).alias("d"))
       .select("d.symbol", "d.price", "d.volume", "d.timestamp")
       .withColumn("event_time", from_unixtime(col("timestamp") / 1000).cast("timestamp"))
       .withColumn("ingest_date", to_date(col("event_time")))
)

ANCHOR = {
    "BTCUSDT": 77000.0, "ETHUSDT": 2300.0, "SOLUSDT": 85.0,
    "BNBUSDT":   630.0, "ADAUSDT":    0.25,
}
LOWER_RATIO, UPPER_RATIO = 0.3, 3.0

anchor_expr = None
for sym, px in ANCHOR.items():
    branch = when(col("symbol") == sym, lit(px))
    anchor_expr = branch if anchor_expr is None else anchor_expr.when(col("symbol") == sym, lit(px))
anchor_expr = anchor_expr.otherwise(lit(None))

clean = (
    parsed.withColumn("anchor_px", anchor_expr)
    .filter(col("price") > 0)
    .filter(col("volume") >= 0)
    .filter(
        col("anchor_px").isNull() |
        ((col("price") >= col("anchor_px") * LOWER_RATIO) &
         (col("price") <= col("anchor_px") * UPPER_RATIO))
    )
    .drop("anchor_px")
)

## 3. Windowed moving average for Gold

Append mode on a streaming aggregation requires a watermark — we already have one. In append mode, each 1-minute window is emitted **once**, after the watermark has moved past its end; the result is immutable and safe to write to Parquet.

In [ ]:
from pyspark.sql.functions import window, avg, count

moving_avg = (
    clean.withWatermark("event_time", "2 minutes")
    .groupBy(
        window(col("event_time"), "1 minute", "10 seconds"),
        col("symbol"),
    )
    .agg(
        avg("price").alias("avg_price"),
        count("*").alias("n_ticks"),
    )
    .select(
        col("symbol"),
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("avg_price"),
        col("n_ticks"),
        to_date(col("window.start")).alias("window_date"),
    )
)

## 4. Launch three writeStream queries → HDFS Parquet

All paths live under `hdfs://namenode:9000/lab3/`. Checkpoints live in the **same HDFS** rather than `/tmp` — this way a crashed job can recover even if the container that produced it is gone.

In [ ]:
HDFS_ROOT = "hdfs://namenode:9000/lab3"

q_bronze = (
    parsed.writeStream
    .format("parquet")
    .option("path",               f"{HDFS_ROOT}/bronze/ticks")
    .option("checkpointLocation", f"{HDFS_ROOT}/checkpoints/bronze")
    .partitionBy("ingest_date")
    .outputMode("append")
    .trigger(processingTime="20 seconds")
    .queryName("bronze_parquet")
    .start()
)

q_silver = (
    clean.writeStream
    .format("parquet")
    .option("path",               f"{HDFS_ROOT}/silver/ticks")
    .option("checkpointLocation", f"{HDFS_ROOT}/checkpoints/silver")
    .partitionBy("ingest_date", "symbol")
    .outputMode("append")
    .trigger(processingTime="20 seconds")
    .queryName("silver_parquet")
    .start()
)

q_gold = (
    moving_avg.writeStream
    .format("parquet")
    .option("path",               f"{HDFS_ROOT}/gold/moving_avg")
    .option("checkpointLocation", f"{HDFS_ROOT}/checkpoints/gold_moving_avg")
    .partitionBy("window_date")
    .outputMode("append")
    .trigger(processingTime="30 seconds")
    .queryName("gold_moving_avg_parquet")
    .start()
)

for q in (q_bronze, q_silver, q_gold):
    print(q.name, "->", q.id, "isActive=", q.isActive)

## 5. Monitor progress

Bronze and Silver begin writing within the first trigger (~20 s). Gold only starts emitting once the watermark has moved past the first complete 1-minute window — plan on ~2–3 minutes before the first Parquet file appears under `gold/`.

In [ ]:
for q in (q_bronze, q_silver, q_gold):
    p = q.lastProgress
    if p:
        print(f"{q.name}: batchId={p['batchId']} "
              f"in/sec={p['inputRowsPerSecond']:.2f} out/sec={p['processedRowsPerSecond']:.2f}")
    else:
        print(f"{q.name}: no progress yet")

## 6. Verify Parquet landed in HDFS

From a shell: `docker exec lab3-namenode hdfs dfs -ls -R /lab3/`.

From this notebook, read back and show a sample:

In [ ]:
# Read back the bronze layer as a batch DataFrame (not streaming) and inspect
bronze_df = spark.read.parquet(f"{HDFS_ROOT}/bronze/ticks")
print("bronze rows:", bronze_df.count())
bronze_df.show(5, truncate=False)

In [ ]:
silver_df = spark.read.parquet(f"{HDFS_ROOT}/silver/ticks")
print("silver rows:", silver_df.count())
silver_df.show(5, truncate=False)

In [ ]:
# Gold only populates after the watermark closes the first window (~2-3 min).
try:
    gold_df = spark.read.parquet(f"{HDFS_ROOT}/gold/moving_avg")
    print("gold rows:", gold_df.count())
    gold_df.orderBy("window_start", "symbol").show(10, truncate=False)
except Exception as e:
    print("Gold not ready yet:", e)

## 7. Clean shutdown

In [ ]:
for q in (q_bronze, q_silver, q_gold):
    q.stop()
print("stopped all queries")

In [ ]:
spark.stop()